# Skill 34: Causal Masking (Look-Ahead Mask)

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Block attention to future positions so a decoder can be trained in parallel without cheating.

## Objectives
- Understand autoregressive generation and information leakage.
- Construct a triangular mask with `torch.tril`.
- Apply the mask before softmax so blocked probabilities become zero.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$A=\mathrm{softmax}(S+M),\quad M_{ij}=\begin{cases}0&j\le i\\-\infty&j>i\end{cases}$$
The mask goes *before* softmax so the surviving weights renormalise.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import math

T, D = 5, 8
x = torch.randn(1, T, D)
q, k, v = x, x, x
# True entries indicate future positions to block.
blocked = ~torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = q @ k.transpose(-2, -1) / math.sqrt(D)
scores = scores.masked_fill(blocked, float("-inf"))
weights = scores.softmax(dim=-1)
print("blocked mask:\n", blocked.int())
print("attention weights:\n", weights[0].round(decimals=3))
assert torch.allclose(weights[0].triu(1), torch.zeros_like(weights[0].triu(1)))


## What to check
**Expected result:** All weights strictly above the diagonal are zero; each row sums to 1 (the `assert` checks it). Inputs are random and unseeded, so numbers vary.

## Pitfalls
Do not invert the mask; do not mask after softmax; combine with padding masks by logical OR and never block an entire row.